In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Labs"
)

xpt_file = DATA_DIR / "GHB_J.xpt"
csv_file = DATA_DIR / "GHB_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2017-2018 GHB_J XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "LBXGH"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "GHB_J column names/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 6401:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 2:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,401 rows x 2 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# WHOLE-NUMBER:
#
#     SEQN
#         respondent identifier
#
#
# GENUINE DECIMAL:
#
#     LBXGH
#         Glycohemoglobin / HbA1c (%)
#
#
# IMPORTANT:
#
# LBXGH is a real decimal laboratory measurement.
#
# Examples:
#
#     5.4
#     6.1
#     7.8
#
# Therefore:
#
#     DO NOT convert LBXGH to Int64
#     DO NOT round LBXGH
#     DO NOT force float_format
#

INTEGER_COLUMNS = [
    "SEQN"
]


DECIMAL_COLUMNS = [
    "LBXGH"
]


if set(
    INTEGER_COLUMNS
    +
    DECIMAL_COLUMNS
) != set(EXPECTED_COLUMNS):

    raise ValueError(
        "Field-type plan does not cover "
        "all GHB_J variables."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


# ============================================================
# 5. VERIFY NUMERIC SOURCE STRUCTURE
# ============================================================

numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected both GHB_J variables "
        "to load numerically.\n\n"
        f"Actual numeric columns:\n{numeric_cols}"
    )


print(
    "PASS: Both GHB_J fields loaded numerically."
)


# ============================================================
# 6. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas decoding artifact:
#
#     5.397605346934028e-79
#
# Some NHANES files contain this where zero was intended.
#
# Direct inspection of attached GHB_J:
#
#     expected tiny count = 0
#
# Therefore absolutely NO zero restoration should occur.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (
    df[EXPECTED_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- ORIGINAL XPT TINY-VALUE CHECK ---"
)

print(
    tiny_counts.to_string()
)

print(
    "Total tiny artifacts:",
    tiny_total
)


if tiny_total != 0:

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in GHB_J.\n\n"
        + tiny_counts[
            tiny_counts > 0
        ].to_string()
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 7. CHECK ORIGINAL NUMERIC ZERO PATTERN
# ============================================================

zero_counts = (
    df[EXPECTED_COLUMNS]
    .eq(0)
    .sum()
)


zero_total = int(
    zero_counts.sum()
)


print(
    "\n--- ORIGINAL NUMERIC ZERO CHECK ---"
)

print(
    zero_counts.to_string()
)

print(
    "Total ordinary numeric zeros:",
    zero_total
)


if zero_total != 0:

    raise ValueError(
        "Unexpected numeric zeros found "
        "in attached GHB_J."
    )


print(
    "PASS: Original GHB_J contains no numeric zeros."
)


# ============================================================
# 8. EXPECTED CORRECTED SOURCE
# ============================================================

#
# GHB_J requires NO numeric repair.
#
# Therefore:
#
#     expected_df == source_df
#

expected_df = source_df.copy()


# ============================================================
# 9. FRACTIONAL VALUE DISCOVERY
# ============================================================

#
# Run BEFORE converting SEQN to Int64.
#
# np.round() below is ONLY used as a comparison test.
#
# It DOES NOT change any source value.
#

print(
    "\n--- FRACTIONAL VALUE DISCOVERY ---"
)


fractional_counts = {}


for col in EXPECTED_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_count = 0

    else:

        fractional_mask = ~np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )

        fractional_count = int(
            fractional_mask.sum()
        )


    fractional_counts[col] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


EXPECTED_FRACTIONAL_COUNTS = {
    "SEQN": 0,
    "LBXGH": 5485
}


if fractional_counts != EXPECTED_FRACTIONAL_COUNTS:

    raise ValueError(
        "Unexpected fractional-value pattern.\n\n"
        f"Expected:\n{EXPECTED_FRACTIONAL_COUNTS}\n\n"
        f"Actual:\n{fractional_counts}"
    )


print(
    "PASS: LBXGH correctly identified "
    "as a genuine decimal variable."
)


# ============================================================
# 10. VERIFY SEQN IS WHOLE-NUMBER
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


seqn_whole_mask = np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
)


seqn_fractional_count = int(
    (~seqn_whole_mask)
    .sum()
)


print(
    "\n--- SEQN INTEGER VALIDATION ---"
)

print(
    "Fractional SEQN observations:",
    seqn_fractional_count
)


if seqn_fractional_count != 0:

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


print(
    "PASS: SEQN is a proven whole-number field."
)


# ============================================================
# 11. VALIDATE SEQN
# ============================================================

print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93705:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 93705-102956 validated."
)


# ============================================================
# 12. CDC / ATTACHMENT CHECK - LBXGH
# ============================================================

#
# CDC published:
#
#     Glycohemoglobin (%)
#
#     Range:
#         3.8 - 16.2
#
#     Nonmissing:
#         6,045
#
#     Missing:
#         356
#

gh_nonmissing = int(
    df["LBXGH"]
    .notna()
    .sum()
)


gh_missing = int(
    df["LBXGH"]
    .isna()
    .sum()
)


gh_min = float(
    df["LBXGH"]
    .min()
)


gh_max = float(
    df["LBXGH"]
    .max()
)


print(
    "\n--- CDC CHECK: LBXGH ---"
)

print(
    "Nonmissing:",
    f"{gh_nonmissing:,}"
)

print(
    "Missing:",
    f"{gh_missing:,}"
)

print(
    "Minimum:",
    gh_min
)

print(
    "Maximum:",
    gh_max
)


if gh_nonmissing != 6045:

    raise ValueError(
        "LBXGH nonmissing count differs from CDC."
    )


if gh_missing != 356:

    raise ValueError(
        "LBXGH missing count differs from CDC."
    )


if not np.isclose(
    gh_min,
    3.8,
    rtol=0,
    atol=0
):

    raise ValueError(
        "LBXGH minimum differs from CDC."
    )


if not np.isclose(
    gh_max,
    16.2,
    rtol=0,
    atol=0
):

    raise ValueError(
        "LBXGH maximum differs from CDC."
    )


print(
    "PASS: LBXGH matches CDC range/counts."
)


# ============================================================
# 13. LBXGH TENTH-PERCENT GRID CHECK
# ============================================================

#
# The released HbA1c values occur on a tenth-percent grid.
#
# Examples:
#
#     5.4
#     6.0
#     7.2
#
# This is a CHECK ONLY.
#
# Nothing is rounded or rewritten.
#

gh_values = (
    df["LBXGH"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


tenth_grid_mask = np.isclose(
    gh_values * 10,
    np.round(
        gh_values * 10
    ),
    rtol=0,
    atol=1e-10
)


tenth_grid_failures = int(
    (~tenth_grid_mask)
    .sum()
)


print(
    "\n--- LBXGH DECIMAL GRID CHECK ---"
)

print(
    "Values not on tenth-percent grid:",
    tenth_grid_failures
)


if tenth_grid_failures != 0:

    raise ValueError(
        "Unexpected LBXGH decimal precision."
    )


print(
    "PASS: All LBXGH values are on "
    "the expected tenth-percent grid."
)


# ============================================================
# 14. WHOLE VS FRACTIONAL LBXGH DIAGNOSTIC
# ============================================================

#
# LBXGH is a decimal VARIABLE even when an individual
# value happens to equal:
#
#     5.0
#     6.0
#
# Therefore whole-valued observations must NOT cause the
# variable to be converted to integer.
#

gh_whole_mask = np.isclose(
    gh_values,
    np.round(gh_values),
    rtol=0,
    atol=1e-12
)


gh_whole_count = int(
    gh_whole_mask.sum()
)


gh_fractional_count = int(
    (~gh_whole_mask)
    .sum()
)


print(
    "\n--- LBXGH DECIMAL DIAGNOSTIC ---"
)

print(
    "Whole-valued LBXGH observations:",
    gh_whole_count
)

print(
    "Fractional LBXGH observations:",
    gh_fractional_count
)


if gh_whole_count != 560:

    raise ValueError(
        "Unexpected whole-valued LBXGH count."
    )


if gh_fractional_count != 5485:

    raise ValueError(
        "Unexpected fractional LBXGH count."
    )


print(
    "PASS: LBXGH variable-level decimal status confirmed."
)


# ============================================================
# 15. VERIFY LBXGH IS EXACTLY UNCHANGED
# ============================================================

#
# No transformations have been permitted on LBXGH.
#

original_gh = (
    source_df["LBXGH"]
    .to_numpy(
        dtype=float
    )
)


current_gh = (
    df["LBXGH"]
    .to_numpy(
        dtype=float
    )
)


if not np.allclose(
    original_gh,
    current_gh,
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "LBXGH changed before export."
    )


print(
    "PASS: LBXGH remains exactly unchanged."
)


# ============================================================
# 16. CONVERT ONLY SEQN TO Int64
# ============================================================

#
# Only SEQN is converted.
#
# LBXGH remains float.
#

df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\nPASS: SEQN converted to nullable Int64."
)

print(
    "PASS: LBXGH retained as genuine decimal."
)


# ============================================================
# 17. COMPLETE PRE-EXPORT VALUE FIDELITY
# ============================================================

#
# GHB_J permitted numeric changes:
#
#     NONE
#
# SEQN type representation changes, but its numeric
# value must remain identical.
#

print(
    "\n--- COMPLETE PRE-EXPORT VALUE FIDELITY ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: All GHB_J numeric values remain unchanged."
)


# ============================================================
# 18. MISSING-VALUE FIDELITY BEFORE EXPORT
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    source_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    current_missing = (
        df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            source_missing
            !=
            current_missing
        )
    )


print(
    "\nMissing-position differences before export:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 19. FINAL ZERO CHECK BEFORE EXPORT
# ============================================================

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


if final_zero_counts != {}:

    raise ValueError(
        "Unexpected numeric zeros introduced:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No numeric zeros introduced."
)


# ============================================================
# 20. EXPORT CSV
# ============================================================

#
# CRITICAL FIDELITY POLICY
# ------------------------------------------------------------
#
# DO NOT:
#
#     df.round(...)
#
# DO NOT:
#
#     float_format=
#
# DO NOT:
#
#     convert LBXGH to Int64
#
# DO NOT:
#
#     force one decimal place
#
#
# The goal is NUMERIC fidelity.
#
# The same export policy protects genuine decimals in:
#
#     Dietary
#     Laboratory
#     Examination
#     survey weights
#     nutrients
#     environmental measurements
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 21. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

#
# IMPORTANT:
#
# Exact floating-point fidelity should be checked using:
#
#     float_precision="round_trip"
#
# Do NOT use pd.to_numeric(raw CSV text) for exact
# binary-float equality.
#

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 22. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric fidelity differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: Both GHB_J variables round-trip exactly."
)


# ============================================================
# 23. CSV MISSING-VALUE FIDELITY
# ============================================================

csv_missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    csv_missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- CSV MISSING-VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    csv_missing_mismatches
)


if csv_missing_mismatches != 0:

    raise ValueError(
        "Missing positions changed during CSV export."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 24. READ FINAL CSV AS RAW TEXT
# ============================================================

#
# Raw text is used ONLY for:
#
#     .0 formatting
#     scientific notation
#     literal zero checks
#
# It is NOT used for exact float fidelity.
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


if raw_csv_df.shape != (
    6401,
    2
):

    raise ValueError(
        "Raw CSV dimensions changed."
    )


if (
    raw_csv_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "Raw CSV column order changed."
    )


# ============================================================
# 25. SEQN .0 CHECK
# ============================================================

print(
    "\n--- SEQN .0 CHECK ---"
)


seqn_dot_zero_count = int(
    (
        raw_csv_df["SEQN"]
        .ne("")
        &
        raw_csv_df["SEQN"]
        .str.endswith(".0")
    )
    .sum()
)


print(
    "SEQN values ending '.0':",
    seqn_dot_zero_count
)


if seqn_dot_zero_count != 0:

    raise ValueError(
        "SEQN still contains unnecessary '.0'."
    )


print(
    "PASS: SEQN contains no '.0'."
)


# ============================================================
# 26. LBXGH RAW DECIMAL CHECK
# ============================================================

#
# DO NOT apply the SEQN .0 rule to LBXGH.
#
# LBXGH is a genuine decimal variable.
#
# A value may legitimately serialize as:
#
#     5.0
#
# That is allowed.
#

gh_tokens = (
    raw_csv_df["LBXGH"]
)


gh_nonblank_tokens = (
    gh_tokens[
        gh_tokens != ""
    ]
)


gh_dot_zero_tokens = int(
    gh_nonblank_tokens
    .str.endswith(".0")
    .sum()
)


print(
    "\n--- LBXGH RAW DECIMAL CHECK ---"
)

print(
    "Populated LBXGH tokens:",
    len(gh_nonblank_tokens)
)

print(
    "LBXGH tokens ending '.0' "
    "(allowed):",
    gh_dot_zero_tokens
)


if len(gh_nonblank_tokens) != 6045:

    raise ValueError(
        "LBXGH populated CSV count changed."
    )


print(
    "PASS: LBXGH retained without forced formatting."
)


# ============================================================
# 27. LBXGH EXACT ROUND-TRIP FIDELITY
# ============================================================

expected_gh_values = (
    expected_df["LBXGH"]
    .to_numpy(
        dtype=float
    )
)


final_gh_values = (
    roundtrip_df["LBXGH"]
    .to_numpy(
        dtype=float
    )
)


gh_exact = np.allclose(
    expected_gh_values,
    final_gh_values,
    rtol=0,
    atol=0,
    equal_nan=True
)


print(
    "\n--- LBXGH EXACT FIDELITY ---"
)

print(
    "Exact XPT -> CSV -> float round-trip:",
    gh_exact
)


if not gh_exact:

    raise ValueError(
        "LBXGH changed during CSV export."
    )


print(
    "PASS: LBXGH decimal values preserved exactly."
)


# ============================================================
# 28. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY-VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 29. FINAL ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts != {}:

    raise ValueError(
        "Unexpected numeric zeros in final CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no numeric zeros."
)


# ============================================================
# 30. SCIENTIFIC-NOTATION CHECK
# ============================================================

scientific_errors = {}


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    count = int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


    if count > 0:

        scientific_errors[col] = count


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    scientific_errors
)


if scientific_errors:

    raise ValueError(
        "Unexpected scientific notation found:\n"
        + str(scientific_errors)
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 31. FINAL SEQN VALIDATION
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93705:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains missing SEQN."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN 93705-102956 valid "
    "and contains no '.0'."
)


# ============================================================
# 32. FINAL ALL-COLUMN EXACT FIDELITY
# ============================================================

#
# Use roundtrip_df for exact numeric comparisons.
#

print(
    "\n--- FINAL ALL-COLUMN SOURCE -> CSV FIDELITY ---"
)


final_fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        final_fidelity_failures.append(
            col
        )


if final_fidelity_failures:

    raise ValueError(
        "Final source -> CSV fidelity failures:\n"
        + str(final_fidelity_failures)
    )


print(
    "PASS: Final CSV preserves all original "
    "GHB_J numeric values exactly."
)


# ============================================================
# 33. ROUND-TRIP MAXIMUM DIFFERENCE DIAGNOSTIC
# ============================================================

#
# Expected:
#
#     0.0
#
# for both columns.
#

print(
    "\n--- ROUND-TRIP MAXIMUM ABSOLUTE DIFFERENCES ---"
)


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    valid_mask = (
        ~np.isnan(expected_values)
        &
        ~np.isnan(final_values)
    )


    if valid_mask.any():

        max_difference = float(
            np.max(
                np.abs(
                    expected_values[valid_mask]
                    -
                    final_values[valid_mask]
                )
            )
        )

    else:

        max_difference = 0.0


    print(
        f"{col}: {max_difference}"
    )


# ============================================================
# 34. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "GHB_J CONVERSION COMPLETE"
)

print(
    "=" * 90
)

print(
    "Cycle: 2017-2018"
)

print(
    "Component: Laboratory - Glycohemoglobin"
)

print(
    "Variable: LBXGH (HbA1c %)"
)

print(
    "Eligible population: "
    "Examined participants age 12+"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny-value artifacts:",
    tiny_total
)

print(
    "Numeric zero restorations:",
    0
)

print(
    "Other numeric corrections:",
    0
)

print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "LBXGH nonmissing:",
    gh_nonmissing
)

print(
    "LBXGH missing:",
    gh_missing
)

print(
    "LBXGH range:",
    f"{gh_min}-{gh_max}"
)

print(
    "LBXGH whole-valued observations:",
    gh_whole_count
)

print(
    "LBXGH fractional observations:",
    gh_fractional_count
)

print(
    "Exact decimal round-trip failures: "
    "0 expected"
)

print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)

print(
    "Missing-position differences: "
    "0 expected"
)

print(
    "Tiny values remaining: "
    "0 expected"
)

print(
    "Unnecessary '.0' in SEQN: "
    "0 expected"
)

print(
    "Scientific notation: "
    "0 expected"
)

print(
    "=" * 90
)

NHANES 2017-2018 GHB_J XPT -> CSV
Rows: 6,401
Columns: 2
PASS: Structure = 6,401 rows x 2 columns.

--- FIELD TYPE PLAN ---
Whole-number fields: ['SEQN']
Genuine decimal fields: ['LBXGH']
PASS: Both GHB_J fields loaded numerically.

--- ORIGINAL XPT TINY-VALUE CHECK ---
SEQN     0
LBXGH    0
Total tiny artifacts: 0
PASS: No tiny XPORT artifacts.

--- ORIGINAL NUMERIC ZERO CHECK ---
SEQN     0
LBXGH    0
Total ordinary numeric zeros: 0
PASS: Original GHB_J contains no numeric zeros.

--- FRACTIONAL VALUE DISCOVERY ---
SEQN: 0 fractional observations
LBXGH: 5,485 fractional observations
PASS: LBXGH correctly identified as a genuine decimal variable.

--- SEQN INTEGER VALIDATION ---
Fractional SEQN observations: 0
PASS: SEQN is a proven whole-number field.

--- SEQN ---
Minimum: 93705.0
Maximum: 102956.0
Missing: 0
Duplicates: 0
PASS: SEQN 93705-102956 validated.

--- CDC CHECK: LBXGH ---
Nonmissing: 6,045
Missing: 356
Minimum: 3.8
Maximum: 16.2
PASS: LBXGH matches CDC range/counts.

--- 